# Ollama Vision Model (VLM) Image Analysis

This notebook demonstrates how to send an image along with a prompt to a local Ollama server running a Vision-Language Model (VLM) using only Python's standard libraries.

## Step 1: Read and Encode the Image

First, we read the local image file (`curry_rice.jpg`) in binary mode and encode it into a Base64 string so it can be transmitted via JSON.
<img src="curry_rice.jpg" alt="Curry Rice" width="300">

In [ ]:
import base64

image_path = 'curry_rice.jpg'

try:
    with open(image_path, 'rb') as f:
        image_base64 = base64.b64encode(f.read()).decode('utf-8')
    print(f'Successfully loaded and encoded: {image_path}')
except FileNotFoundError:
    print(f'Error: Image file not found at {image_path}')

## Step 2: Prepare and POST the Request to Ollama

Next, we construct the API payload containing the model name, text prompt, and the Base64-encoded image, then send a POST request to the local Ollama server.

> **Note:** `gemma4:12b` is a Vision-Language Model (VLM), meaning it can directly interpret and reason about Base64-encoded image data provided in the `images` field alongside the text prompt.

In [ ]:
import json
import urllib.request

url = 'http://localhost:11434/api/chat'
headers = {"Content-Type": "application/json"}

payload = {
    "model": "gemma4:12b",
    "messages": [
        {
            "role": "user",
            "content": "Here is a photo. Briefly describe the characteristics of this Japanese-style curry rice.",
            "images": [image_base64],
        }
    ],
    "stream": False,
}

try:
    req_data = json.dumps(payload).encode('utf-8')
    req = urllib.request.Request(url, data=req_data, headers=headers, method='POST')
    print('Request payload prepared successfully.')
except Exception as e:
    print(f'Error preparing request: {e}')

## Step 3: Receive and Print the JSON Output

Finally, we execute the request, read the response from the Ollama server, parse the JSON body, and print the formatted result (handling any potential connection or file errors gracefully).

In [ ]:
try:
    with urllib.request.urlopen(req) as response:
        res_body = response.read().decode('utf-8')
        res_json = json.loads(res_body)
        content = res_json.get('message', {}).get('content', '')
        print(content)
        print('=====')
        print(json.dumps(res_json, ensure_ascii=False, indent=2))

except urllib.error.URLError as e:
    error_response = {
        'error': True,
        'message': f'Failed to connect to Ollama server: {e.reason}',
    }
    print(json.dumps(error_response, ensure_ascii=False, indent=2))
except Exception as e:
    error_response = {'error': True, 'message': str(e)}
    print(json.dumps(error_response, ensure_ascii=False, indent=2))